# 🐦 AlphaFinch quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shloksobti/alphafinch/blob/main/examples/quickstart.ipynb)

AlphaFinch lets an AI **invent trading strategies**, breeds the best ones over generations, then makes them **sit an exam on data they never saw**. This notebook runs in your browser. No setup, and no API key until the last (optional) step.

1. Install
2. Watch evolution offline (no AI, no network)
3. Write your own strategy and backtest it on Indian stocks
4. See what it would hold today, with and without shorting
5. Re-run the world exam on the strategy that passed it
6. *(Optional)* Let an AI breed strategies for you

Repo: [github.com/shloksobti/alphafinch](https://github.com/shloksobti/alphafinch) · Guide: [docs/guide.md](https://github.com/shloksobti/alphafinch/blob/main/docs/guide.md)

## 1. Install

In [ ]:
!pip install -q alphafinch
!alphafinch

## 2. Evolution, offline

A synthetic market, no AI and no network. Strategies are bred with parameter tweaks and blends, and the champion sits the sealed exam at the end. Takes about a minute.

In [ ]:
!alphafinch demo --generations 6

## 3. Write a strategy and backtest it

A strategy is one function: given prices (and more `data`), return the fraction of capital to hold in each stock, every day. The toolkit `tk` is available without importing.

This one is sector-neutral momentum: within each sector, buy the strongest six-month performers and short the weakest, rebalancing monthly.

The first run downloads NIFTY 200 data (a few minutes, then cached).

In [ ]:
%%writefile my_strategy.py
def strategy(prices, data):
    """Sector Spread: sector-neutral 6-month momentum. Hypothesis: news diffuses slowly within industries."""
    LOOKBACK = 126
    score = tk.neutralize(tk.zscore(prices.pct_change(LOOKBACK)), data.sector)
    return tk.rebalance(tk.long_short(score, q=0.2), every="M")

In [ ]:
!alphafinch backtest my_strategy.py india

This simple idea **loses** a little after costs (look at the alpha and the later eras), which is typical: most textbook ideas stopped working long ago. That's why AlphaFinch breeds thousands of variants, and why it seals data away to judge them. `backtest` only uses the training years, so the most recent three stay sealed for a fair exam.

## 4. What would it hold today?

The same strategy, but under **long-only** rules (no shorting, as in the cash market). The engine enforces the rules, whatever the strategy asks for.

In [ ]:
!alphafinch holdings my_strategy.py india --long-only --top 8

## 5. The strategy that passed

*Quiet Sector Tether* was bred by the AI on US data before 2017, frozen, and then tested on 7 stock markets it had never seen: the UK, Eurozone, Japan, Hong Kong, Australia, Canada and Korea, 2017–2026. This cell re-runs that pre-registered test exactly. `--prior-looks 4` counts the other four strategies tested alongside it, so the bar is the registered 2.33. It downloads 7 markets on the first run (a few minutes).

In [ ]:
!curl -sL -o qst.py https://raw.githubusercontent.com/shloksobti/alphafinch/main/forward/world/us-s12-champion-ebe31d7c2f.py
!alphafinch world-exam qst.py --prior-looks 4

You should see alpha in every market and a pooled **PASS** (t ≈ 3.4). The full write-up, including the caveats and the four strategies that didn't pass, is in [docs/preregistration-world.md](https://github.com/shloksobti/alphafinch/blob/main/docs/preregistration-world.md).

## 6. (Optional) Let an AI breed strategies

Add an API key in Colab's **🔑 Secrets** panel (left sidebar) as `ANTHROPIC_API_KEY` or `OPENAI_API_KEY`, then run. A short run is a few minutes and a few dozen AI calls.

In [ ]:
import os
try:
    from google.colab import userdata
    for k in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY"):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
except ImportError:
    pass
print("AI key found" if any(os.environ.get(k) for k in ("ANTHROPIC_API_KEY", "OPENAI_API_KEY")) else "No key: add one in Secrets, or skip this step")

In [ ]:
!pip install -q "alphafinch[all]"
!alphafinch evolve india --generations 3 --islands 2 --no-reveal

Each run writes `runs/<timestamp>/report.html` (the champion, its family tree, the lab notebook and the exam verdict) and `champion.py`, ready for `backtest`, `holdings` or `world-exam`.

If AlphaFinch was useful or fun, a ⭐ on [GitHub](https://github.com/shloksobti/alphafinch) helps others find it.

*Research software, not investment advice.*